# 🩺 Diabetix — Knowledge Distillation Fine-Tuning
**Model:** Qwen2.5-7B-Instruct → LoRA fine-tuned
**Hardware:** Kaggle T4×2
**Framework:** Unsloth + TRL
**Dataset:** 600 synthetic diabetes Q&A examples (T1D + T2D)

---
### Pipeline Overview
```
1. Install dependencies
2. Load base model (4-bit quantized)
3. Load & validate dataset
4. Configure LoRA
5. Train
6. Evaluate
7. Save & export
```

## Cell 1 — Install Dependencies

In [1]:
# Takes ~3-4 minutes on Kaggle
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps trl peft accelerate bitsandbytes
!pip install --quiet datasets sentencepiece protobuf

print("✅ Dependencies installed")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.7 MB/s eta 0:00:00:00:01:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 2.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 76.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━

## Cell 2 — GPU Check

In [15]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")

# T4 has 15GB VRAM each — 4-bit Qwen2.5-7B needs ~5GB, plenty of room
print("\n✅ GPU check passed")

CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 | VRAM: 15.6 GB
  GPU 1: Tesla T4 | VRAM: 15.6 GB

✅ GPU check passed


## Cell 3 — Load Base Model (Qwen2.5-7B-Instruct, 4-bit)

In [16]:
from unsloth import FastLanguageModel

# ── Config ──────────────────────────────────────────────
MODEL_NAME     = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # pre-quantized, fastest load
MAX_SEQ_LEN    = 2048   # covers 30-day logs + reasoning + output comfortably
LOAD_IN_4BIT   = True
DTYPE          = None   # auto-detect (bfloat16 on T4)
# ────────────────────────────────────────────────────────

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

print(f"✅ Loaded: {MODEL_NAME}")
print(f"   Max sequence length: {MAX_SEQ_LEN} tokens")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Loaded: unsloth/Qwen2.5-7B-Instruct-bnb-4bit
   Max sequence length: 2048 tokens


## Cell 4 — Apply LoRA

In [4]:
# ── LoRA Config ─────────────────────────────────────────
# r=32 is stronger than default r=16 — better for reasoning distillation
# Targets all attention + MLP layers for maximum expressiveness
LORA_R         = 32
LORA_ALPHA     = 64    # alpha = 2×r is standard best practice
LORA_DROPOUT   = 0.05
BIAS           = "none"
# ────────────────────────────────────────────────────────

model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    target_modules      = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",       # MLP
    ],
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    bias                = BIAS,
    use_gradient_checkpointing = "unsloth",  # saves VRAM
    random_state        = 42,
    use_rslora          = True,   # rank-stabilized LoRA — better convergence
    loftq_config        = None,
)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied")
print(f"   Trainable params: {trainable:,} ({100*trainable/total:.2f}% of total)")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.7 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA applied
   Trainable params: 80,740,352 (1.82% of total)


## Cell 5 — Define Chat Template

In [19]:
# Qwen2.5 uses ChatML format — we wrap our fields into it
# Training target = REASONING + OUTPUT together (chain-of-thought distillation)

SYSTEM_PROMPT = """You are Diabetix, a specialized diabetes pattern analysis assistant.
You help patients understand their glucose patterns, meal impacts, and receive personalized advisory.
You support both Type 1 and Type 2 diabetes management.
Always reason step by step before answering. Never fabricate values not present in the logs.
For any insulin dose suggestion, always include a disclaimer that it is a mathematical estimate only."""

def format_example(example):
    """
    Formats a dataset example into ChatML format for training.
    Expected fields: profile, input, reasoning, output
    """
    user_content = f"""{example['profile']}

{example['input']}"""

    assistant_content = f"""<reasoning>
{example['reasoning']}
</reasoning>

<answer>
{example['output']}
</answer>"""

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

print("✅ Chat template defined")
print("   Training format: system + user(profile+logs+question) + assistant(reasoning+output)")

✅ Chat template defined
   Training format: system + user(profile+logs+question) + assistant(reasoning+output)


## Cell 6 — Load & Validate Dataset

> **Before running this cell:**
> Upload your generated JSONL files to Kaggle as a dataset.
> Each line must be a valid JSON object with these fields:
> `profile`, `input`, `reasoning`, `output`, `category`, `diabetes_type`
>
> Expected files:
> - `/kaggle/input/diabetix-dataset/analysis_600.jsonl`
> - OR combine both categories into one file

In [17]:
import json
from datasets import Dataset
from collections import Counter

# ── Config ──────────────────────────────────────────────
DATASET_PATH = "/kaggle/input/datasets/yacineabh/q-and-a-analysis/diabetix_training_data.jsonl"
# ────────────────────────────────────────────────────────

def load_jsonl(path):
    examples = []
    errors = []
    required_fields = {"profile", "input", "reasoning", "output", "category", "diabetes_type"}

    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                missing = required_fields - set(obj.keys())
                if missing:
                    errors.append(f"Line {i+1}: missing fields {missing}")
                    continue
                # Basic length check — reject suspiciously short reasoning
                if len(obj["reasoning"]) < 100:
                    errors.append(f"Line {i+1}: reasoning too short ({len(obj['reasoning'])} chars)")
                    continue
                examples.append(obj)
            except json.JSONDecodeError as e:
                errors.append(f"Line {i+1}: JSON parse error — {e}")

    return examples, errors

raw_examples, errors = load_jsonl(DATASET_PATH)
# After loading raw_examples, add:
raw_examples = [ex for ex in raw_examples if ex["category"] == "analysis"]
print(f"Filtered to analysis only: {len(raw_examples)} examples")

# Report
print(f"✅ Loaded {len(raw_examples)} valid examples")
if errors:
    print(f"⚠️  {len(errors)} errors found:")
    for e in errors[:5]:
        print(f"   {e}")

# Distribution check
categories = Counter(ex["category"] for ex in raw_examples)
diabetes_types = Counter(ex["diabetes_type"] for ex in raw_examples)
print(f"\n📊 Category distribution: {dict(categories)}")
print(f"📊 Diabetes type distribution: {dict(diabetes_types)}")

Filtered to analysis only: 400 examples
✅ Loaded 400 valid examples

📊 Category distribution: {'analysis': 400}
📊 Diabetes type distribution: {'Type 1': 185, 'Type 2': 215}


## Cell 7 — Tokenize & Check Lengths

In [20]:
import numpy as np

# Format all examples
formatted = [format_example(ex) for ex in raw_examples]
dataset = Dataset.from_list(formatted)

# Check token lengths to catch examples that exceed MAX_SEQ_LEN
lengths = []
for ex in formatted:
    tokens = tokenizer(ex["text"], return_tensors="pt")
    lengths.append(tokens["input_ids"].shape[1])

lengths = np.array(lengths)
over_limit = np.sum(lengths > MAX_SEQ_LEN)

print(f"Token length stats:")
print(f"  Min:    {lengths.min()}")
print(f"  Max:    {lengths.max()}")
print(f"  Mean:   {lengths.mean():.0f}")
print(f"  p95:    {np.percentile(lengths, 95):.0f}")
print(f"  > {MAX_SEQ_LEN} tokens: {over_limit} examples ({100*over_limit/len(lengths):.1f}%)")

if over_limit > 0:
    print(f"\n⚠️  {over_limit} examples exceed MAX_SEQ_LEN and will be truncated.")
    print("   Consider increasing MAX_SEQ_LEN or shortening those examples.")
else:
    print("\n✅ All examples within token limit")

Token length stats:
  Min:    834
  Max:    2779
  Mean:   1532
  p95:    2118
  > 2048 tokens: 26 examples (6.5%)

⚠️  26 examples exceed MAX_SEQ_LEN and will be truncated.
   Consider increasing MAX_SEQ_LEN or shortening those examples.


## Cell 8 — Train / Validation Split

In [8]:
# 90% train / 10% validation
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"✅ Split complete")
print(f"   Train: {len(train_dataset)} examples")
print(f"   Eval:  {len(eval_dataset)} examples")

✅ Split complete
   Train: 360 examples
   Eval:  40 examples


In [23]:
%%writefile train.py
import os
import json
import torch
import numpy as np
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments

os.environ["TOKENIZERS_PARALLELISM"] = "false"
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# 1. LOAD BASE MODEL
local_rank = int(os.environ.get("LOCAL_RANK", "0"))
torch.cuda.set_device(local_rank) # 👉 ADD THIS LINE: Locks the process to the specific GPU

# 1. LOAD BASE MODEL
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    device_map = {"": local_rank},
)

# ... (the rest of the script stays exactly the same)

# 2. APPLY LORA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = True,
)

# 3. PREPARE DATASET
SYSTEM_PROMPT = """You are Diabetix, a specialized diabetes pattern analysis assistant.
You help patients understand their glucose patterns, meal impacts, and receive personalized advisory.
You support both Type 1 and Type 2 diabetes management.
Always reason step by step before answering. Never fabricate values not present in the logs.
For any insulin dose suggestion, always include a disclaimer that it is a mathematical estimate only."""

def format_example(example):
    user_content = f"{example['profile']}\n\n{example['input']}"
    assistant_content = f"<reasoning>\n{example['reasoning']}\n</reasoning>\n\n<answer>\n{example['output']}\n</answer>"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

# Load from your Kaggle path
DATASET_PATH = "/kaggle/input/datasets/yacineabh/q-and-a-analysis/diabetix_training_data.jsonl"
raw_examples = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            raw_examples.append(json.loads(line.strip()))

formatted = [format_example(ex) for ex in raw_examples]
dataset = Dataset.from_list(formatted)

# Split 90/10
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

# 4. TRAINING ARGUMENTS (Optimized for 2 GPUs)
training_args = TrainingArguments(
    output_dir                  = "/kaggle/working/diabetix-checkpoints",
    num_train_epochs            = 3,
    per_device_train_batch_size = 1,     # 2 per GPU
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = 8,     # Effective batch = 2 (batch) * 4 (accum) * 2 (GPUs) = 16
    warmup_steps                = 10,    # Changed from warmup_ratio to avoid warnings
    learning_rate               = 2e-4,
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 10,
    eval_strategy               = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 50,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    gradient_checkpointing = True, # 👈 ADD THIS LINE
    report_to                   = "none",
    ddp_find_unused_parameters  = False, # MUST BE FALSE FOR MULTI-GPU LORA
    seed                        = 42,
)

# 5. TRAIN
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = eval_dataset,
    dataset_text_field = "text",
    max_seq_length     = 2048,
    args               = training_args,
)

print("🚀 Starting Multi-GPU Training...")

local_rank = int(os.environ.get("LOCAL_RANK", "0"))
target_device = torch.device(f"cuda:{local_rank}")
for name, param in model.named_parameters():
    if param.device.type == "cpu":
        param.data = param.data.to(target_device)

# 2. Sweep and move any stranded buffers (like RoPE inv_freq)
for name, buffer in model.named_buffers():
    if buffer.device.type == "cpu":
        buffer.data = buffer.data.to(target_device)



trainer.train()

# 6. SAVE
ADAPTER_SAVE_PATH = "/kaggle/working/diabetix-lora-adapter"
trainer.model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"✅ Adapter saved to {ADAPTER_SAVE_PATH}")

Overwriting train.py


In [24]:
!pip install xformers
!torchrun --nproc_per_node=2 train.py

W0525 02:26:16.095000 2889 torch/distributed/run.py:852] 
W0525 02:26:16.095000 2889 torch/distributed/run.py:852] *****************************************
W0525 02:26:16.095000 2889 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0525 02:26:16.095000 2889 torch/distributed/run.py:852] *****************************************
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUD

In [27]:
from unsloth import FastLanguageModel
import torch

# ── 1. Load Model ────────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "/kaggle/working/diabetix-lora-adapter",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)

# ── 2. System Prompt ─────────────────────────────────────────
SYSTEM_PROMPT = """You are Diabetix, a specialized diabetes pattern analysis assistant.

STRICT GROUNDING RULES — NEVER VIOLATE:
- Only reference glucose values, meals, and insulin doses explicitly present in the provided logs
- Never invent frequency counts unless that exact number of days appear in the logs
  (e.g. if only 1 day is provided, never say "3 out of 3 days")
- Never suggest specific glucose values for foods not mentioned in the logs
- Never add advice unrelated to what the logs show (e.g. no random glucagon mentions)
- If logs are insufficient to answer confidently, say so explicitly

REASONING RULES:
- Always compute real deltas: post-meal glucose minus pre-meal glucose
- Always compute implied ICR: carbs ÷ insulin units (result is g/u, NOT mg/dL/u)
- Count exact occurrences of each meal/pattern across the days provided
- Distinguish T1D vs T2D reasoning:
    T1D → insulin timing, ICR accuracy, stacking risk, active insulin overlap
    T2D → carb load, insulin resistance, meal GI, medication timing, lifestyle levers

RESPONSE FORMAT — ALWAYS:
- Reason step by step inside <reasoning> tags
- Output structured JSON inside <answer> tags
- For any insulin dose suggestion: add disclaimer inside the JSON suggestion field"""

# ── 3. Inference Function ────────────────────────────────────
def run_inference(profile: str, input_text: str, max_new_tokens: int = 700) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"{profile}\n\n{input_text}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.15,   # lower = less hallucination
            do_sample          = True,
            top_p              = 0.9,
            repetition_penalty = 1.15,   # stronger = less invention
            pad_token_id       = tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response  = tokenizer.decode(generated, skip_special_tokens=True)
    return response


def pretty_print(response: str):
    """Splits reasoning and answer blocks for readable output."""
    print("\n" + "═"*60)

    if "<reasoning>" in response and "</reasoning>" in response:
        reasoning = response.split("<reasoning>")[1].split("</reasoning>")[0].strip()
        print("🧠 REASONING:")
        print("─"*60)
        print(reasoning)
    else:
        print("⚠️  No <reasoning> block found")

    if "<answer>" in response and "</answer>" in response:
        answer = response.split("<answer>")[1].split("</answer>")[0].strip()
        print("\n💊 ANSWER:")
        print("─"*60)
        # Try to pretty-print JSON
        try:
            import json
            parsed = json.loads(answer)
            print(json.dumps(parsed, indent=2))
        except json.JSONDecodeError:
            print(answer)
    else:
        print("⚠️  No <answer> block found — raw output:")
        print(response)

    print("═"*60 + "\n")


# ── 4. Test Cases ────────────────────────────────────────────

# TEST 1: T1D with full 15-day logs (proper test)
profile_t1d = "Diabetes Type: Type 1\nRegimen: MDI (Novolog + Tresiba)"

logs_15d = """--- DAY -14 ---
[07:30] Glucose: 98
[07:45] Meal: Oatmeal + banana (~55g carbs)
[07:45] Insulin: 5u rapid
[09:45] Glucose: 185 (Spike)
[13:00] Glucose: 110
[13:15] Meal: Chicken salad & quinoa (~40g carbs)
[13:15] Insulin: 4u rapid
[15:30] Glucose: 115
[19:00] Glucose: 122
[19:15] Meal: Steak and roasted potatoes (~60g carbs)
[19:15] Insulin: 6u rapid
[21:30] Glucose: 142

--- DAY -13 ---
[08:15] Glucose: 104
[08:30] Meal: 3 Scrambled eggs + 1 slice whole toast (~15g carbs)
[08:30] Insulin: 1.5u rapid
[10:30] Glucose: 112 (Flat profile)
[13:00] Glucose: 101
[13:15] Meal: Leftover steak salad (~10g carbs)
[13:15] Insulin: 1u rapid
[18:45] Glucose: 118
[19:00] Meal: Grilled salmon + brown rice (~45g carbs)
[19:00] Insulin: 4.5u rapid
[21:00] Glucose: 135

--- DAY -12 ---
[07:45] Glucose: 95
[08:00] Meal: Oatmeal + banana (~55g carbs)
[08:00] Insulin: 5u rapid
[10:00] Glucose: 179 (Spike)
[13:00] Glucose: 120
[13:20] Meal: Sushi roll platter (~70g carbs)
[13:25] Insulin: 7u rapid
[16:00] Glucose: 155
[19:30] Glucose: 130
[19:45] Meal: Roasted chicken breast & broccoli (~15g carbs)
[19:45] Insulin: 1.5u rapid
[22:00] Glucose: 110

--- DAY -11 ---
[08:00] Glucose: 101
[08:15] Meal: Greek yogurt + berries + almonds (~20g carbs)
[08:15] Insulin: 2u rapid
[10:15] Glucose: 118
[13:00] Glucose: 99
[13:15] Meal: Turkey and swiss wrap (~35g carbs)
[13:15] Insulin: 3.5u rapid
[15:00] Glucose: 124
[19:00] Glucose: 112
[19:15] Meal: Turkey chili (~40g carbs)
[19:15] Insulin: 4u rapid
[21:30] Glucose: 138

--- DAY -10 ---
[07:55] Glucose: 106
[08:05] Meal: Oatmeal + banana (~55g carbs)
[08:05] Insulin: 5u rapid
[10:05] Glucose: 174 (Spike)
[13:15] Glucose: 115
[13:30] Meal: Chicken Caesar wrap (~45g carbs)
[13:30] Insulin: 4.5u rapid
[15:30] Glucose: 122
[19:00] Glucose: 119
[19:15] Meal: Pasta with marinara sauce (~75g carbs)
[19:15] Insulin: 9u rapid
[21:15] Glucose: 168 (Spike)

--- DAY -9 ---
[08:20] Glucose: 112
[08:35] Meal: Omelet with spinach & feta (~5g carbs)
[08:35] Insulin: 0.5u rapid
[10:35] Glucose: 114
[13:00] Glucose: 102
[13:10] Meal: Mixed greens with tuna & olive oil (~8g carbs)
[13:10] Insulin: 1u rapid
[18:30] Glucose: 98
[18:45] Meal: Deep dish pepperoni pizza (~85g carbs, high fat)
[18:45] Insulin: 8u rapid
[20:45] Glucose: 125 (Normal early check)
[22:45] Glucose: 198 (Delayed high fat absorption spike)

--- DAY -8 ---
[08:10] Glucose: 118 (High fasting due to yesterday pizza)
[08:20] Meal: Oatmeal + banana (~55g carbs)
[08:20] Insulin: 5u rapid
[10:20] Glucose: 182 (Spike)
[13:00] Glucose: 110
[13:15] Meal: Beef burger on lettuce wrap + side salad (~12g carbs)
[13:15] Insulin: 1u rapid
[15:30] Glucose: 105
[19:00] Glucose: 100
[19:15] Meal: Shrimp stir-fry with white rice (~65g carbs)
[19:15] Insulin: 6.5u rapid
[21:15] Glucose: 148

--- DAY -7 ---
[07:45] Glucose: 100
[08:00] Meal: 2 Eggs + 1 slice buttered toast (~15g carbs)
[08:00] Insulin: 1.5u rapid
[10:00] Glucose: 108
[14:00] Glucose: 92
[14:15] Meal: Homemade pesto pasta (~75g carbs)
[14:15] Insulin: 9u rapid (Overestimated active profile)
[16:00] Glucose: 65 (Mild Hypoglycemia)
[16:05] Action: Consumed 15g fast-acting carbs (Apple juice)
[16:20] Glucose: 95"""

print("TEST 1 — T1D oatmeal pattern (15 days)\n")
response1 = run_inference(profile_t1d, logs_15d + "\n\nWhy does oatmeal always spike me even with 5u?")
pretty_print(response1)


# TEST 2: T2D advisory trigger
profile_t2d = "Diabetes Type: Type 2\nRegimen: Metformin 1000mg twice daily + Lantus 18u nightly"

logs_t2d = """--- DAY -5 ---
[08:00] Glucose: 142
[08:15] Meal: White bread + jam (~50g carbs)
[10:15] Glucose: 218 (Spike)
[13:00] Glucose: 158
[13:15] Meal: Rice + grilled chicken (~65g carbs)
[15:30] Glucose: 192 (Spike)
[19:00] Glucose: 145
[19:15] Meal: Lentil soup + bread (~55g carbs)
[21:15] Glucose: 188 (Spike)

--- DAY -4 ---
[08:10] Glucose: 138
[08:20] Meal: Oats + berries (~35g carbs)
[10:20] Glucose: 162
[13:00] Glucose: 128
[13:15] Meal: Grilled chicken + salad (~15g carbs)
[15:30] Glucose: 134
[19:00] Glucose: 122
[19:15] Meal: Baked salmon + vegetables (~18g carbs)
[21:15] Glucose: 138

--- DAY -3 ---
[08:00] Glucose: 135
[08:15] Meal: Cornflakes + milk (~55g carbs)
[10:15] Glucose: 224 (Spike)
[13:00] Glucose: 160
[13:20] Meal: Pasta salad (~60g carbs)
[15:30] Glucose: 198 (Spike)
[19:15] Glucose: 140
[19:30] Meal: Grilled chicken + broccoli (~12g carbs)
[21:30] Glucose: 148

[ADVISORY TRIGGER]"""

print("\nTEST 2 — T2D advisory (auto-triggered)\n")
response2 = run_inference(profile_t2d, logs_t2d)
pretty_print(response2)

==((====))==  Unsloth 2026.5.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 1 — T1D oatmeal pattern (15 days)



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=700) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



════════════════════════════════════════════════════════════
🧠 REASONING:
────────────────────────────────────────────────────────────
Oatmeal appeared Day -14 (-105), -12 (-179), -10 (-174). Pre-meal avg: (98+104+95)/3 = 100. Avg delta: (185+179+174) / 3 - 100 = +78 mg/dL. Same dose, same food, similar pre-meal range. Implied ICR: 55g/5u = ~11g per u. High-carb breakfasts (Day -14, -12, -10): all spiked >165 despite identical dose. Pizza Day -9: 85g carbs at 8pm, 8u insulin, ended up 198 at 10pm — delayed fat-laden absorption. Low-fat lunch Day -8: beef burger + side salad, 12g carbs, 1u insulin, stayed flat 105. The issue isn't the dose or the food itself; it's the combination of high-carb content AND NO FAT. Fat slows gastric emptying and extends carbohydrate absorption over hours rather than minutes. Without fat, oatmeal hits the stomach fully liquid and absorbs within 15 min. This patient needs either a small fat addition OR a larger insulin dose split into two halves (pre-bolus 

In [29]:
# 1. Delete the giant failed GGUF folder to free up ~15GB of space
!rm -rf /kaggle/working/diabetix-q4km

# 2. Delete any incomplete zip file from the previous attempt
!rm -f diabetix_adapter.zip

# 3. Zip the adapter again now that the disk can breathe
!zip -r diabetix_adapter.zip /kaggle/working/diabetix-lora-adapter

print("✅ Junk cleared and adapter zipped successfully!")

  adding: kaggle/working/diabetix-lora-adapter/ (stored 0%)
  adding: kaggle/working/diabetix-lora-adapter/tokenizer.json (deflated 81%)
  adding: kaggle/working/diabetix-lora-adapter/adapter_config.json (deflated 57%)
  adding: kaggle/working/diabetix-lora-adapter/adapter_model.safetensors (deflated 8%)
  adding: kaggle/working/diabetix-lora-adapter/README.md (deflated 65%)
  adding: kaggle/working/diabetix-lora-adapter/chat_template.jinja (deflated 71%)
  adding: kaggle/working/diabetix-lora-adapter/tokenizer_config.json (deflated 89%)
✅ Junk cleared and adapter zipped successfully!


In [26]:
GGUF_PATH = "/kaggle/working/diabetix-q4km"

print("🚀 Starting GGUF quantization (this takes a few minutes)...")
model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer,
    quantization_method="q4_k_m"   # Best quality/size tradeoff
)
print(f"✅ GGUF Q4_K_M model saved to: {GGUF_PATH}")

🚀 Starting GGUF quantization (this takes a few minutes)...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/diabetix-q4km/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:48<00:00, 12.18s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [02:08<00:00, 32.19s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/diabetix-q4km`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...


RuntimeError: Unsloth: GGUF conversion failed in Kaggle environment.
This is likely due to the 20GB disk space limit.
Try saving to /tmp directory or use a smaller model.
Error: Unsloth: Failed to convert model to GGUF with command `/usr/bin/python3 /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile Qwen2.5-7B-Instruct.F16.gguf --outtype f16 --split-max-size 50G /kaggle/working/diabetix-q4km`: Command '['/usr/bin/python3', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py', '--outfile', 'Qwen2.5-7B-Instruct.F16.gguf', '--outtype', 'f16', '--split-max-size', '50G', '/kaggle/working/diabetix-q4km']' returned non-zero exit status 1.

## Cell 9 — Training Config & Launch

Estimated time on T4×2:
- 600 samples × 3 epochs = 1800 steps at batch 2 + grad accum 4
- **~2.5 to 3.5 hours**

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# ── Training Config ──────────────────────────────────────
OUTPUT_DIR      = "/kaggle/working/diabetix-checkpoints"
NUM_EPOCHS      = 3
BATCH_SIZE      = 2      # per device (T4 has 15GB, 4-bit model ~5GB → safe)
GRAD_ACCUM      = 4      # effective batch = 2×4 = 8
WARMUP_RATIO    = 0.05
LR              = 2e-4   # standard for LoRA r=32
LR_SCHEDULER    = "cosine"
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 1.0
SAVE_STEPS      = 50
EVAL_STEPS      = 50
LOGGING_STEPS   = 10
# ────────────────────────────────────────────────────────

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_ratio                = WARMUP_RATIO,
    learning_rate               = LR,
    lr_scheduler_type           = LR_SCHEDULER,
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = MAX_GRAD_NORM,
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = LOGGING_STEPS,
    eval_strategy         = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 3,            # keep only last 3 checkpoints
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",       # disable wandb
    dataloader_num_workers      = 2,
    # group_by_length             = True,         # batches similar lengths → faster
    seed                        = 42,
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    packing         = False,    # OFF — our examples are long enough already
    args            = training_args,
)

print("✅ Trainer configured")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Steps per epoch: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"   Total steps: ~{NUM_EPOCHS * len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")
print("\n🚀 Starting training...")

trainer_stats = trainer.train()

print(f"\n✅ Training complete")
print(f"   Total time: {trainer_stats.metrics['train_runtime']/60:.1f} minutes")
print(f"   Final train loss: {trainer_stats.metrics['train_loss']:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/360 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/40 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


✅ Trainer configured
   Effective batch size: 8
   Epochs: 3
   Steps per epoch: 45
   Total steps: ~135

🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 360 | Num Epochs = 3 | Total steps = 69
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


OutOfMemoryError: CUDA out of memory. Tried to allocate 448.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 449.81 MiB is free. Including non-PyTorch memory, this process has 14.12 GiB memory in use. Of the allocated memory 13.88 GiB is allocated by PyTorch, and 95.95 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Cell 10 — Loss Curve

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps  = [x["step"] for x in log_history if "loss" in x and "eval_loss" not in x]
train_losses = [x["loss"] for x in log_history if "loss" in x and "eval_loss" not in x]
eval_steps   = [x["step"] for x in log_history if "eval_loss" in x]
eval_losses  = [x["eval_loss"] for x in log_history if "eval_loss" in x]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_steps, train_losses, label="Train Loss", color="steelblue", linewidth=1.5)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(eval_steps, eval_losses, label="Eval Loss", color="coral", linewidth=1.5, marker="o", markersize=4)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/loss_curve.png", dpi=150)
plt.show()
print("✅ Loss curve saved")

# Quick overfitting check
if eval_losses:
    gap = eval_losses[-1] - train_losses[-1]
    if gap > 0.3:
        print("⚠️  Possible overfitting detected (eval loss >> train loss). Consider reducing epochs.")
    else:
        print(f"✅ Train/eval loss gap: {gap:.3f} — looks healthy")

## Cell 11 — Qualitative Evaluation (Sanity Check)

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)  # enable native 2x faster inference

TEST_CASES = [
    {
        "label": "T1D — oatmeal spike pattern",
        "profile": "Diabetes Type: Type 1\nRegimen: MDI (Novolog + Tresiba)",
        "input": """--- DAY -3 ---
[08:00] Glucose: 98
[08:10] Meal: Oatmeal + banana (~55g carbs)
[08:10] Insulin: 5u rapid
[10:10] Glucose: 184 (Spike)

--- DAY -2 ---
[07:50] Glucose: 101
[08:00] Meal: Oatmeal + banana (~55g carbs)
[08:00] Insulin: 5u rapid
[10:00] Glucose: 179 (Spike)

--- DAY -1 ---
[08:15] Glucose: 3 eggs + toast (~15g carbs)
[08:15] Insulin: 1.5u rapid
[10:15] Glucose: 112 (Flat)

Why does oatmeal always spike me even with 5u?"""
    },
    {
        "label": "T2D — advisory trigger",
        "profile": "Diabetes Type: Type 2\nRegimen: Metformin + Basal insulin (Lantus 20u nightly)",
        "input": """--- DAY -5 ---
[08:00] Glucose: 142
[08:15] Meal: White bread + jam (~50g carbs)
[10:15] Glucose: 218
[13:00] Glucose: 155
[13:15] Meal: Rice + fried chicken (~70g carbs)
[15:30] Glucose: 198

--- DAY -4 ---
[08:10] Glucose: 138
[08:20] Meal: Oats + berries (~35g carbs)
[10:20] Glucose: 162
[13:00] Glucose: 128
[13:15] Meal: Grilled chicken + salad (~15g carbs)
[15:30] Glucose: 134

[ADVISORY TRIGGER]"""
    },
]

def run_inference(profile, input_text, max_new_tokens=600):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"{profile}\n\n{input_text}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens   = max_new_tokens,
        temperature      = 0.2,
        do_sample        = True,
        repetition_penalty = 1.1,
        pad_token_id     = tokenizer.eos_token_id,
    )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


for tc in TEST_CASES:
    print(f"\n{'='*60}")
    print(f"TEST: {tc['label']}")
    print('='*60)
    response = run_inference(tc["profile"], tc["input"])
    print(response)
    print()

## Cell 12 — Save LoRA Adapter

In [ ]:
ADAPTER_SAVE_PATH = "/kaggle/working/diabetix-lora-adapter"

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"✅ LoRA adapter saved to: {ADAPTER_SAVE_PATH}")
print("   This is a lightweight adapter (~150-200MB), NOT the full model")
print("   Load it on top of the base Qwen2.5-7B model for inference")

# List saved files
import os
for f in os.listdir(ADAPTER_SAVE_PATH):
    size = os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, f))
    print(f"   {f}: {size/1e6:.1f} MB")

## Cell 13 — Export Merged Model for Ollama (GGUF)

In [ ]:
# Option A: Save merged model in float16 (for vLLM or HuggingFace inference)
MERGED_PATH = "/kaggle/working/diabetix-merged-f16"
model.save_pretrained_merged(
    MERGED_PATH,
    tokenizer,
    save_method="merged_16bit"
)
print(f"✅ Merged float16 model saved to: {MERGED_PATH}")

# Option B: Export to GGUF Q4_K_M (for Ollama — recommended for local use)
# This quantizes the merged model to ~4.5GB — runs on any machine with 8GB RAM
GGUF_PATH = "/kaggle/working/diabetix-q4km"
model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer,
    quantization_method="q4_k_m"   # best quality/size tradeoff
)
print(f"✅ GGUF Q4_K_M model saved to: {GGUF_PATH}")
print("   Upload the .gguf file to HuggingFace or use directly with Ollama")

## Cell 14 — Load with Ollama (run locally after download)

After downloading the `.gguf` file, run these commands on your local machine:

```bash
# 1. Create a Modelfile
cat > Modelfile << 'EOF'
FROM ./diabetix-q4km.gguf
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
SYSTEM "You are Diabetix, a specialized diabetes pattern analysis assistant. You support both Type 1 and Type 2 diabetes management. Always reason step by step. Never fabricate values not in the logs. For any insulin dose suggestion, include a disclaimer."
EOF

# 2. Create the Ollama model
ollama create diabetix -f Modelfile

# 3. Run it
ollama run diabetix
```

Or use it via Python:
```python
import ollama

response = ollama.chat(
    model="diabetix",
    messages=[{"role": "user", "content": your_prompt}],
    options={"temperature": 0.2}
)
print(response["message"]["content"])
```

## Cell 15 — Upload Adapter to HuggingFace Hub (optional)

Keep your model private — add `private=True`

In [ ]:
# Optional — uncomment if you want to push to HuggingFace

# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

# model.push_to_hub(
#     "your-username/diabetix-qwen25-7b-lora",
#     tokenizer=tokenizer,
#     private=True
# )
# print("✅ Uploaded to HuggingFace Hub (private)")

print("HuggingFace upload — uncomment lines above and add your token to use")

---
## Summary

| Step | What happened |
|------|---------------|
| Base model | Qwen2.5-7B-Instruct, 4-bit quantized |
| LoRA config | r=32, alpha=64, RSLoRA, all attention+MLP layers |
| Training | 3 epochs, effective batch 8, cosine LR decay |
| Output | LoRA adapter + merged f16 + GGUF Q4_K_M for Ollama |
| Est. time | 2.5–3.5 hours on T4×2 |

### What to check before calling it done
1. ✅ Eval loss should be decreasing (not flat or rising)
2. ✅ Train/eval loss gap < 0.3 (no overfitting)
3. ✅ Qualitative test responses show step-by-step reasoning
4. ✅ Responses stay within `<reasoning>` + `<answer>` tags
5. ✅ Insulin suggestions always include disclaimer